# 2.3 · Linked lists

*Sequences* · 28 exercises

Build singly and doubly linked lists, walk them with pointers, and see where they beat arrays.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-2.3) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-2.3) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-2.3)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 2.3.1 · Node and LinkedList skeleton  `Core`

**Task.** Create `Node(data)` with `data` and `next`, and `LinkedList` with `head`. Add `__len__` and `to_list()` that walk the chain, plus a `from_list(values)` class method for tests.

**Example.** `LinkedList.from_list([1, 2, 3]).to_list()` -> `[1, 2, 3]`

**Big O.** `__len__` and `to_list` are O(N): there is no length stored, so every node must be visited.

**Note.** Nodes can live anywhere in memory; each one only knows where the next one is. That is what makes reading slow and insertion at the head fast.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-1)

In [ ]:
# 2.3.1 Node and LinkedList skeleton
class Node:
    def __init__(self, data, next_node=None):
        raise NotImplementedError


class LinkedList:
    def __init__(self, head=None):
        raise NotImplementedError

    @classmethod
    def from_list(cls, values):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

    def to_list(self):
        raise NotImplementedError

In [ ]:
# Test helpers: run once.
L = LinkedList.from_list

In [ ]:
def test_ex_2_3_1():
    assert L([1, 2, 3]).to_list() == [1, 2, 3]
    assert len(L([1, 2, 3])) == 3
    assert len(L([])) == 0
    lst = L([7])
    assert lst.head.data == 7
    assert lst.head.next is None


test_ex_2_3_1()
print('2.3.1 passed')

### 2.3.2 · Read by index  `Core`

**Task.** Add `read(index)`: follow `next` links `index` times. Return `None` if the list is too short.

**Example.** `[10, 20, 30]` -> `read(2)` is `30`; `read(5)` is `None`

**Big O.** O(N). An array reads in O(1); this is the linked list's biggest cost.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-2)

In [ ]:
# 2.3.2 Read by index
def read(self, index):
    raise NotImplementedError


LinkedList.read = read

In [ ]:
def test_ex_2_3_2():
    lst = L([10, 20, 30])
    assert lst.read(2) == 30
    assert lst.read(5) is None
    assert lst.read(0) == 10


test_ex_2_3_2()
print('2.3.2 passed')

### 2.3.3 · Search for a value  `Core`

**Task.** Add `index_of(value)` returning the position of the first match, or `None`.

**Example.** `[10, 20, 30]` -> `index_of(20)` is `1`

**Big O.** O(N), the same as linear search in an array.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-3)

In [ ]:
# 2.3.3 Search for a value
def index_of(self, value):
    raise NotImplementedError


LinkedList.index_of = index_of

In [ ]:
def test_ex_2_3_3():
    lst = L([10, 20, 30])
    assert lst.index_of(20) == 1
    assert lst.index_of(99) is None


test_ex_2_3_3()
print('2.3.3 passed')

### 2.3.4 · Insert at the start  `Core`

**Task.** Add `insert_at_start(value)`: the new node points at the old head and becomes the head.

**Example.** `[2, 3]` -> `insert_at_start(1)` -> `[1, 2, 3]`

**Big O.** O(1). The array's worst case is the linked list's best case.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-4)

In [ ]:
# 2.3.4 Insert at the start
def insert_at_start(self, value):
    raise NotImplementedError


LinkedList.insert_at_start = insert_at_start

In [ ]:
def test_ex_2_3_4():
    lst = L([2, 3])
    lst.insert_at_start(1)
    assert lst.to_list() == [1, 2, 3]
    lst = L([])
    lst.insert_at_start(1)
    assert lst.to_list() == [1]


test_ex_2_3_4()
print('2.3.4 passed')

### 2.3.5 · Insert at the end  `Core`

**Task.** Add `insert_at_end(value)`: walk to the last node and attach the new one. Handle the empty list.

**Example.** `[1, 2]` -> `insert_at_end(3)` -> `[1, 2, 3]`

**Big O.** O(N) without a tail pointer (see 2.3.15 for O(1)).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-5)

In [ ]:
# 2.3.5 Insert at the end
def insert_at_end(self, value):
    raise NotImplementedError


LinkedList.insert_at_end = insert_at_end

In [ ]:
def test_ex_2_3_5():
    lst = L([1, 2])
    lst.insert_at_end(3)
    assert lst.to_list() == [1, 2, 3]
    lst = L([])
    lst.insert_at_end(1)
    assert lst.to_list() == [1]


test_ex_2_3_5()
print('2.3.5 passed')

### 2.3.6 · Insert at an index  `Core`

**Task.** Add `insert_at(index, value)`: walk to the node just before `index`, then rewire two links. Raise `IndexError` if `index` is past the end.

**Example.** `[1, 2, 4]` -> `insert_at(2, 3)` -> `[1, 2, 3, 4]`

**Big O.** O(N) to find the spot, then O(1) to insert. No shifting, unlike an array.

**Note.** Set `new_node.next` before changing `prev.next`, or the rest of the list is lost.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-6)

In [ ]:
# 2.3.6 Insert at an index
def insert_at(self, index, value):
    raise NotImplementedError


LinkedList.insert_at = insert_at

In [ ]:
def test_ex_2_3_6():
    lst = L([1, 2, 4])
    lst.insert_at(2, 3)
    assert lst.to_list() == [1, 2, 3, 4]
    lst.insert_at(4, 5)
    lst.insert_at(0, 0)
    assert lst.to_list() == [0, 1, 2, 3, 4, 5]
    with pytest.raises(IndexError):
        L([1]).insert_at(3, 9)


test_ex_2_3_6()
print('2.3.6 passed')

### 2.3.7 · Delete at the start  `Core`

**Task.** Add `delete_at_start()` returning the removed value (or `None` if empty): move `head` to `head.next`.

**Example.** `[1, 2, 3]` -> returns `1`, leaves `[2, 3]`

**Big O.** O(1). Arrays need O(N) here.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-7)

In [ ]:
# 2.3.7 Delete at the start
def delete_at_start(self):
    raise NotImplementedError


LinkedList.delete_at_start = delete_at_start

In [ ]:
def test_ex_2_3_7():
    lst = L([1, 2, 3])
    assert lst.delete_at_start() == 1
    assert lst.to_list() == [2, 3]
    assert L([]).delete_at_start() is None


test_ex_2_3_7()
print('2.3.7 passed')

### 2.3.8 · Delete at the end  `Core`

**Task.** Add `delete_at_end()` returning the removed value: find the second-to-last node and cut its link. Handle 0 and 1 nodes.

**Example.** `[1, 2, 3]` -> returns `3`, leaves `[1, 2]`

**Big O.** O(N) in a singly linked list, because you need the node before the last one.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-8)

In [ ]:
# 2.3.8 Delete at the end
def delete_at_end(self):
    raise NotImplementedError


LinkedList.delete_at_end = delete_at_end

In [ ]:
def test_ex_2_3_8():
    lst = L([1, 2, 3])
    assert lst.delete_at_end() == 3
    assert lst.to_list() == [1, 2]
    lst = L([1])
    assert lst.delete_at_end() == 1
    assert lst.to_list() == []


test_ex_2_3_8()
print('2.3.8 passed')

### 2.3.9 · Delete at an index  `Core`

**Task.** Add `delete_at(index)` returning the removed value: point the previous node's `next` past the doomed node. Raise `IndexError` if out of range.

**Example.** `[1, 2, 9, 3]` -> `delete_at(2)` returns `9`, leaves `[1, 2, 3]`

**Big O.** O(N) to reach it, O(1) to unlink.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-9)

In [ ]:
# 2.3.9 Delete at an index
def delete_at(self, index):
    raise NotImplementedError


LinkedList.delete_at = delete_at

In [ ]:
def test_ex_2_3_9():
    lst = L([1, 2, 9, 3])
    assert lst.delete_at(2) == 9
    assert lst.to_list() == [1, 2, 3]
    assert lst.delete_at(2) == 3
    assert lst.to_list() == [1, 2]
    with pytest.raises(IndexError):
        lst.delete_at(2)
    with pytest.raises(IndexError):
        L([]).delete_at(0)


test_ex_2_3_9()
print('2.3.9 passed')

### 2.3.10 · Delete by value  `Core`

**Task.** Add `delete_value(value)` that removes the first matching node and returns `True`, or `False`.

**Example.** `[4, 5, 4]` -> `delete_value(4)` -> `[5, 4]`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-10) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-10)

In [ ]:
# 2.3.10 Delete by value
def delete_value(self, value):
    raise NotImplementedError


LinkedList.delete_value = delete_value

In [ ]:
def test_ex_2_3_10():
    lst = L([4, 5, 4])
    assert lst.delete_value(4)
    assert lst.to_list() == [5, 4]
    assert lst.delete_value(4)
    assert lst.to_list() == [5]
    assert not lst.delete_value(7)


test_ex_2_3_10()
print('2.3.10 passed')

### 2.3.11 · Split a list into halves  `Practice`

**Task.** Add `split_half()`: cut the list after its middle node (the first half keeps the extra node when the count is odd) and return a new `LinkedList` holding the second half. Find the cut with a slow pointer that moves one step and a fast pointer that moves two.

**Example.** `[1, 2, 3, 4, 5]` -> the list keeps `[1, 2, 3]`, the result is `[4, 5]`

**Big O.** O(N) time, O(1) extra space: one walk with two pointers, and no nodes are copied.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-11) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-11)

In [ ]:
# 2.3.11 Split a list into halves
def split_half(self):
    raise NotImplementedError


LinkedList.split_half = split_half

In [ ]:
def test_ex_2_3_11():
    lst = L([1, 2, 3, 4, 5])
    second = lst.split_half()
    assert lst.to_list() == [1, 2, 3]
    assert second.to_list() == [4, 5]
    even = L([1, 2, 3, 4])
    assert even.split_half().to_list() == [3, 4]
    assert even.to_list() == [1, 2]
    one, empty = (L([7]), L([]))
    assert one.split_half().to_list() == []
    assert one.to_list() == [7]
    assert empty.split_half().to_list() == []


test_ex_2_3_11()
print('2.3.11 passed')

### 2.3.12 · Reverse in place  `Core`

**Task.** Add `reverse()` that flips every `next` link using `previous`, `current` and `next` variables.

**Example.** `[1, 2, 3]` -> `[3, 2, 1]`

**Big O.** O(N) time, O(1) space.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-12) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-12)

In [ ]:
# 2.3.12 Reverse in place
def reverse(self):
    raise NotImplementedError


LinkedList.reverse = reverse

In [ ]:
def test_ex_2_3_12():
    lst = L([1, 2, 3])
    lst.reverse()
    assert lst.to_list() == [3, 2, 1]
    empty = L([])
    empty.reverse()
    assert empty.to_list() == []


test_ex_2_3_12()
print('2.3.12 passed')

### 2.3.14 · Detect a cycle  `Stretch`

**Task.** Add `has_cycle()` using Floyd's slow/fast pointers: if they ever meet, the list loops.

**Example.** A list whose last node points back to its second node -> `True`

**Big O.** O(N) time, O(1) space (a set of visited nodes would be O(N) space).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-14) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-14)

In [ ]:
# 2.3.14 Detect a cycle
def has_cycle(self):
    raise NotImplementedError


LinkedList.has_cycle = has_cycle

In [ ]:
def test_ex_2_3_14():
    cyc = L([1, 2, 3, 4])
    assert not cyc.has_cycle()
    cyc.head.next.next.next.next = cyc.head.next
    assert cyc.has_cycle()
    assert not L([]).has_cycle()


test_ex_2_3_14()
print('2.3.14 passed')

### 2.3.13 · Merge two sorted lists by relinking  `Practice`

**Task.** Write `merge_sorted_lists(a, b)` for two `LinkedList`s sorted in increasing order, returning a `LinkedList` that holds every node of both in order. Relink the existing nodes (a placeholder node at the front keeps the code short); do not create a node per value.

**Example.** `[1, 4, 7]`, `[2, 3, 9]` -> `[1, 2, 3, 4, 7, 9]`

**Big O.** O(N + M) time, O(1) extra space: the existing nodes are relinked, never copied.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-13) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-13)

In [ ]:
# 2.3.13 Merge two sorted lists by relinking
def merge_sorted_lists(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_13():
    a, b = (L([1, 4, 7]), L([2, 3, 9]))
    first_b = b.head
    merged = merge_sorted_lists(a, b)
    assert merged.to_list() == [1, 2, 3, 4, 7, 9]
    assert merged.head.next is first_b
    assert merge_sorted_lists(L([]), L([5])).to_list() == [5]
    assert merge_sorted_lists(L([2, 2]), L([2])).to_list() == [2, 2, 2]
    assert merge_sorted_lists(L([]), L([])).to_list() == []


test_ex_2_3_13()
print('2.3.13 passed')

### 2.3.15 · Doubly linked list  `Core`

**Task.** Build `DoublyLinkedList` with `head` and `tail` and nodes holding `prev` and `next`. Implement `insert_at_start`, `insert_at_end`, `delete_at_start`, `delete_at_end`, `to_list()` and `to_list_reversed()` (walk back from the tail).

**Example.** insert_at_end 2, insert_at_start 1, insert_at_end 3 -> `[1, 2, 3]`; reversed `[3, 2, 1]`

**Big O.** O(1) at both ends, because the tail pointer and `prev` links remove the walk.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-15) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-15)

In [ ]:
# 2.3.15 Doubly linked list
class DNode:
    def __init__(self, data):
        raise NotImplementedError


class DoublyLinkedList:
    def __init__(self):
        raise NotImplementedError

    def insert_at_start(self, value):
        raise NotImplementedError

    def insert_at_end(self, value):
        raise NotImplementedError

    def delete_at_start(self):
        raise NotImplementedError

    def delete_at_end(self):
        raise NotImplementedError

    def to_list(self):
        raise NotImplementedError

    def to_list_reversed(self):
        raise NotImplementedError

In [ ]:
def test_ex_2_3_15():
    d = DoublyLinkedList()
    d.insert_at_end(2)
    d.insert_at_start(1)
    d.insert_at_end(3)
    assert d.to_list() == [1, 2, 3]
    assert d.to_list_reversed() == [3, 2, 1]
    assert d.delete_at_end() == 3
    assert d.delete_at_start() == 1
    assert d.to_list() == [2]
    assert d.delete_at_end() == 2
    assert d.head is None
    assert d.tail is None
    assert d.delete_at_start() is None


test_ex_2_3_15()
print('2.3.15 passed')

### 2.3.16 · Queue on a doubly linked list  `Core`

**Task.** Build `LinkedQueue` with `enqueue`, `dequeue` and `read` on top of your `DoublyLinkedList`.

**Example.** enqueue 1, 2, 3 -> dequeue `1`, read `2`

**Big O.** O(1) for every operation: insert at the tail, remove from the head.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-16) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-16)

In [ ]:
# 2.3.16 Queue on a doubly linked list
class LinkedQueue:
    def __init__(self):
        raise NotImplementedError

    def enqueue(self, value):
        raise NotImplementedError

    def dequeue(self):
        raise NotImplementedError

    def read(self):
        raise NotImplementedError

In [ ]:
def test_ex_2_3_16():
    q = LinkedQueue()
    for v in (1, 2, 3):
        q.enqueue(v)
    assert q.dequeue() == 1
    assert q.read() == 2


test_ex_2_3_16()
print('2.3.16 passed')

### 2.3.17 · Remove the k-th node from the end  `Core`

**Task.** Write `remove_kth_from_end(lst, k)` that unlinks the `k`-th node from the end of a `LinkedList` (`1 <= k <= len`) in one pass and returns its value: move a lead pointer `k` steps ahead of a trailing one, then move both until the lead reaches the last node. A placeholder node in front handles removing the head.

**Example.** `[1, 2, 3, 4, 5], 2` -> returns `4`; the list is `[1, 2, 3, 5]`

**Big O.** O(N) time, O(1) space, one pass: a lead pointer k nodes ahead reaches the end exactly when the trailing pointer sits before the node to remove.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-17) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-17)

In [ ]:
# 2.3.17 Remove the k-th node from the end
def remove_kth_from_end(lst, k):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_17():
    lst = L([1, 2, 3, 4, 5])
    assert remove_kth_from_end(lst, 2) == 4
    assert lst.to_list() == [1, 2, 3, 5]
    assert remove_kth_from_end(lst, 4) == 1
    assert lst.to_list() == [2, 3, 5]
    one = L([7])
    assert remove_kth_from_end(one, 1) == 7
    assert one.to_list() == []


test_ex_2_3_17()
print('2.3.17 passed')

### 2.3.18 · Add numbers stored as lists  `Practice`

**Task.** Each `LinkedList` holds the digits of a non-negative number, least significant first (`[9, 4, 1]` is 149). Write `add_digit_lists(a, b)` that returns a new `LinkedList` holding the digits of the sum in the same order, walking both lists once with a carry.

**Example.** `[9, 4, 1]`, `[3, 8]` -> `[2, 3, 2]` (149 + 83 = 232)

**Big O.** O(N + M) time and O(max(N, M)) space for the result: one walk down both lists with a carry, like adding on paper.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-18) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-18)

In [ ]:
# 2.3.18 Add numbers stored as lists
def add_digit_lists(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_18():
    assert add_digit_lists(L([9, 4, 1]), L([3, 8])).to_list() == [2, 3, 2]
    assert add_digit_lists(L([9, 9]), L([1])).to_list() == [0, 0, 1]
    assert add_digit_lists(L([]), L([])).to_list() == []
    assert add_digit_lists(L([0]), L([0])).to_list() == [0]
    assert add_digit_lists(L([5]), L([])).to_list() == [5]


test_ex_2_3_18()
print('2.3.18 passed')

### 2.3.19 · Where two lists meet  `Practice`

**Task.** Two `LinkedList`s may share their tail nodes. Write `meeting_node(a, b)` that returns the first node they share (the same object), or `None`, with O(1) extra space: walk one pointer down each list, and when a pointer runs off the end, restart it at the head of the other list.

**Example.** `[1, 2]` then shared `[8, 9]`, and `[5]` then the same `[8, 9]` -> the node holding `8`

**Big O.** O(N + M) time, O(1) space: after switching lists once, both pointers have walked the same distance when they reach the shared part.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-19) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-19)

In [ ]:
# 2.3.19 Where two lists meet
def meeting_node(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_19():
    shared = L([8, 9]).head
    a, b = (LinkedList(Node(1, Node(2, shared))), LinkedList(Node(5, shared)))
    assert meeting_node(a, b) is shared
    assert meeting_node(b, a) is shared
    assert meeting_node(L([1, 2]), L([3])) is None
    assert meeting_node(a, a) is a.head
    assert meeting_node(L([]), b) is None


test_ex_2_3_19()
print('2.3.19 passed')

### 2.3.20 · Palindrome list  `Practice`

**Task.** Write `is_palindrome_list(lst)`: `True` when the values of a `LinkedList` read the same both ways, using O(1) extra space. Find the middle, reverse the second half, compare the halves, and reverse the second half again so the list ends up unchanged.

**Example.** `[1, 2, 3, 2, 1]` -> `True`; `[1, 2]` -> `False`

**Big O.** O(N) time, O(1) space: find the middle with slow and fast pointers, reverse the second half to compare, then reverse it back.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-20) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-20)

In [ ]:
# 2.3.20 Palindrome list
def is_palindrome_list(lst):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_20():
    lst = L([1, 2, 3, 2, 1])
    assert is_palindrome_list(lst)
    assert lst.to_list() == [1, 2, 3, 2, 1]
    lst = L([1, 2, 3, 1])
    assert not is_palindrome_list(lst)
    assert lst.to_list() == [1, 2, 3, 1]
    assert is_palindrome_list(L([1, 2, 2, 1]))
    assert not is_palindrome_list(L([1, 2]))
    assert is_palindrome_list(L([]))
    assert is_palindrome_list(L([4]))


test_ex_2_3_20()
print('2.3.20 passed')

### 2.3.21 · Rotate a list by k  `Core`

**Task.** Write `rotate_right(lst, k)` that rotates a `LinkedList` right by `k` places in place: the last `k` nodes move to the front in order. `k` may exceed the length. Find the length and tail, close the ring, and cut it before the new head.

**Example.** `[1, 2, 3, 4, 5], 2` -> `[4, 5, 1, 2, 3]`

**Big O.** O(N) time, O(1) space: one walk finds the length and tail, a second finds the new tail; three link changes do the rest.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-21) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-21)

In [ ]:
# 2.3.21 Rotate a list by k
def rotate_right(lst, k):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_21():
    for k, want in [(2, [4, 5, 1, 2, 3]), (7, [4, 5, 1, 2, 3]), (0, [1, 2, 3, 4, 5]), (5, [1, 2, 3, 4, 5]), (1, [5, 1, 2, 3, 4])]:
        lst = L([1, 2, 3, 4, 5])
        rotate_right(lst, k)
        assert lst.to_list() == want
    empty = L([])
    rotate_right(empty, 3)
    assert empty.to_list() == []


test_ex_2_3_21()
print('2.3.21 passed')

### 2.3.22 · Interleave with the reversed second half  `Stretch`

**Task.** Write `interleave_ends(lst)` that reorders a `LinkedList` in place to first, last, second, second-to-last, and so on, by relinking nodes: cut the list at its middle, reverse the second half, then alternate nodes from the two halves.

**Example.** `[1, 2, 3, 4, 5]` -> `[1, 5, 2, 4, 3]`; `[1, 2, 3, 4]` -> `[1, 4, 2, 3]`

**Big O.** O(N) time, O(1) space: split at the middle, reverse the second half, then zip the two halves together by relinking.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-22) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-22)

In [ ]:
# 2.3.22 Interleave with the reversed second half
def interleave_ends(lst):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_22():
    for values, want in [([1, 2, 3, 4, 5], [1, 5, 2, 4, 3]), ([1, 2, 3, 4], [1, 4, 2, 3]), ([1, 2], [1, 2]), ([1], [1]), ([], [])]:
        lst = L(values)
        interleave_ends(lst)
        assert lst.to_list() == want


test_ex_2_3_22()
print('2.3.22 passed')

### 2.3.23 · Swap adjacent pairs  `Practice`

**Task.** Write `swap_pairs(lst)` that swaps every two neighbouring nodes of a `LinkedList` in place by relinking them (not by swapping values); an odd last node stays put.

**Example.** `[1, 2, 3, 4, 5]` -> `[2, 1, 4, 3, 5]`

**Big O.** O(N) time, O(1) space: each pair needs three link changes, and a placeholder node in front makes the first pair like the rest.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-23) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-23)

In [ ]:
# 2.3.23 Swap adjacent pairs
def swap_pairs(lst):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_23():
    lst = L([1, 2, 3, 4, 5])
    first = lst.head
    swap_pairs(lst)
    assert lst.to_list() == [2, 1, 4, 3, 5]
    assert lst.head.next is first
    for values, want in [([], []), ([1], [1]), ([1, 2], [2, 1]), ([1, 2, 3, 4], [2, 1, 4, 3])]:
        lst = L(values)
        swap_pairs(lst)
        assert lst.to_list() == want


test_ex_2_3_23()
print('2.3.23 passed')

### 2.3.24 · Mergesort a linked list  `Stretch`

**Task.** Write `sort_list(lst)` that sorts a `LinkedList` in place by relinking its nodes, in O(N log N): split the chain at its middle with slow and fast pointers, sort each half recursively, and merge the two sorted chains (equal values keep their order).

**Example.** `[5, 1, 4, 2, 3, 1]` -> `[1, 1, 2, 3, 4, 5]`

**Big O.** O(N log N) time, O(log N) stack: splitting at the middle costs a walk, and merging relinks nodes, so no buffer is needed.

**Note.** Mergesort suits linked lists better than arrays: the merge needs no extra array, and nothing needs random access.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-24) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-24)

In [ ]:
# 2.3.24 Mergesort a linked list
def sort_list(lst):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_24():
    lst = L([3, 1, 2])
    three = lst.head
    sort_list(lst)
    assert lst.to_list() == [1, 2, 3]
    assert lst.head.next.next is three
    for values in ([5, 1, 4, 2, 3, 1], [], [2, 1], [7], [4, 4, 1, 9, 0, 3, 8]):
        lst = L(values)
        sort_list(lst)
        assert lst.to_list() == sorted(values)


test_ex_2_3_24()
print('2.3.24 passed')

### 2.3.25 · Drop duplicates from a sorted list  `Core`

**Task.** Write `drop_sorted_duplicates(lst)` that unlinks repeated values from a sorted `LinkedList` in place, keeping the first node of each value.

**Example.** `[1, 1, 2, 3, 3, 3]` -> `[1, 2, 3]`

**Big O.** O(N) time, O(1) space: in a sorted list every copy of a value sits next to the first one, so each node is compared once with its successor.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-25) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-25)

In [ ]:
# 2.3.25 Drop duplicates from a sorted list
def drop_sorted_duplicates(lst):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_25():
    for values, want in [([1, 1, 2, 3, 3, 3], [1, 2, 3]), ([], []), ([4, 4], [4]), ([1, 2], [1, 2]), ([0, 0, 0, 5], [0, 5])]:
        lst = L(values)
        drop_sorted_duplicates(lst)
        assert lst.to_list() == want


test_ex_2_3_25()
print('2.3.25 passed')

### 2.3.26 · Partition a list around a value  `Practice`

**Task.** Write `partition_list(lst, x)` that relinks a `LinkedList` in place so every node below `x` comes before every node at or above `x`, keeping the original order within each group. Build two chains behind two placeholder nodes, then join them.

**Example.** `[4, 1, 6, 2, 5, 2], 4` -> `[1, 2, 2, 4, 6, 5]`

**Big O.** O(N) time, O(1) space: nodes are relinked onto two chains, which are then joined.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-26) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-26)

In [ ]:
# 2.3.26 Partition a list around a value
def partition_list(lst, x):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_26():
    for values, x, want in [([4, 1, 6, 2, 5, 2], 4, [1, 2, 2, 4, 6, 5]), ([3, 1], 0, [3, 1]), ([], 5, []), ([5, 6, 1], 9, [5, 6, 1]), ([9, 2, 8, 1], 5, [2, 1, 9, 8])]:
        lst = L(values)
        partition_list(lst, x)
        assert lst.to_list() == want


test_ex_2_3_26()
print('2.3.26 passed')

### 2.3.27 · Flatten a multilevel list  `Stretch`

**Task.** Create `MultiNode(data, next_node=None, child=None)`: a node whose `child` may start another list, which may have children of its own. Write `flatten(head)` that relinks everything into one list through `next`, in place, with each child list inserted right after its parent and before the parent's old `next`; set every `child` to `None` and return `head`. Save pending `next` nodes on a stack.

**Example.** `1-2-3-4`, `2` has child `5-6`, `5` has child `7`, `4` has child `8` -> `[1, 2, 5, 7, 6, 3, 4, 8]`

**Big O.** O(N) time, O(D) space for nesting depth D: each node is visited once, and the stack holds one saved `next` per open level.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-27) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-27)

In [ ]:
# 2.3.27 Flatten a multilevel list
class MultiNode:
    def __init__(self, data, next_node=None, child=None):
        raise NotImplementedError


def flatten(head):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_27():
    n = {v: MultiNode(v) for v in range(1, 9)}
    n[1].next, n[2].next, n[3].next, n[5].next = (n[2], n[3], n[4], n[6])
    n[2].child, n[5].child, n[4].child = (n[5], n[7], n[8])
    values, node = ([], flatten(n[1]))
    while node:
        assert node.child is None
        values.append(node.data)
        node = node.next
    assert values == [1, 2, 5, 7, 6, 3, 4, 8]
    assert flatten(None) is None


test_ex_2_3_27()
print('2.3.27 passed')

### 2.3.28 · Cycle in a number sequence  `Practice`

**Task.** Repeatedly applying `step` to `x0` gives a sequence that must eventually repeat. Write `cycle_start_and_length(step, x0)` that returns `[mu, length]`: the index where the cycle starts and its length, in O(1) space. Use a slow and a fast walker like a linked-list cycle check, then walk again from `x0` to find the entry.

**Example.** `step = lambda x: (x * x + 1) % 11`, `x0 = 0` (sequence `0, 1, 2, 5, 4, 6, 4, ...`) -> `[4, 2]`

**Big O.** O(mu + lambda) steps and O(1) space: the slow and fast walkers meet inside the cycle, and a second walk from the start meets the first at the cycle's entry.

**Note.** The same walk decides whether repeated sums of squared digits ever reach 1: the sequence either settles at 1 or loops.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-3-28) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-3-28)

In [ ]:
# 2.3.28 Cycle in a number sequence
def cycle_start_and_length(step, x0):
    raise NotImplementedError

In [ ]:
def test_ex_2_3_28():
    assert cycle_start_and_length(lambda x: (x * x + 1) % 11, 0) == [4, 2]
    assert cycle_start_and_length(lambda x: (x + 1) % 5, 0) == [0, 5]
    assert cycle_start_and_length(lambda x: x // 2, 20) == [5, 1]
    digit_squares = lambda x: sum((int(d) ** 2 for d in str(x)))
    assert cycle_start_and_length(digit_squares, 7) == [5, 1]
    assert cycle_start_and_length(digit_squares, 4) == [0, 8]


test_ex_2_3_28()
print('2.3.28 passed')